# Ectopy and HRV-Adjusted GEE Analyses

This notebook provides additional analyses to assess whether AF-Mamba predictions are primarily explained by ectopic activity or conventional RR-interval variability measures.

1. **Ectopic burden analysis:** Evaluates AF-Mamba discrimination across low, moderate, and high ectopic-burden groups using pooled out-of-fold predictions and subject-level bootstrap confidence intervals.
2. **HRV-adjusted GEE analysis:** Evaluates the association between AF-Mamba prediction scores and pre-AF status after adjustment for RMSSD, SD1/SD2, SODP-Q1, SODP-CTM100, and PAS.

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv(Path("Results/oof_ectopy_predictions.csv"))

def bootstrap_ci(d, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    af = {sid: g for sid, g in d[d.label==1].groupby("sid")}
    nsr = {sid: g for sid, g in d[d.label==0].groupby("sid")}
    af_ids, nsr_ids = list(af), list(nsr)
    aurocs, auprcs = [], []

    for _ in range(n_boot):
        b = pd.concat(
            [af[sid] for sid in rng.choice(af_ids, len(af_ids), replace=True)] +
            [nsr[sid] for sid in rng.choice(nsr_ids, len(nsr_ids), replace=True)],
            ignore_index=True
        )
        aurocs.append(roc_auc_score(b.label, b.prob))
        auprcs.append(average_precision_score(b.label, b.prob))

    return np.percentile(aurocs, [2.5,97.5]), np.percentile(auprcs, [2.5,97.5])

groups = ["Low (<0.1%)","Moderate (0.1-<0.5%)","High (>=0.5%)","Overall"]
results = []

for group in groups:
    d = df if group=="Overall" else df[df.group==group]
    auroc = roc_auc_score(d.label, d.prob)
    auprc = average_precision_score(d.label, d.prob)
    auc_ci, auprc_ci = bootstrap_ci(d)

    results.append({
        "Ectopy group": group,
        "AF N": (d.label==1).sum(),
        "NSR N": (d.label==0).sum(),
        "AUROC (95% CI)": f"{auroc:.3f} [{auc_ci[0]:.3f}-{auc_ci[1]:.3f}]",
        "AUPRC (95% CI)": f"{auprc:.3f} [{auprc_ci[0]:.3f}-{auprc_ci[1]:.3f}]"
    })

result = pd.DataFrame(results)

print("\n===== AF-MAMBA PERFORMANCE BY ECTOPIC BURDEN =====")
print(result.to_string(index=False))


===== AF-MAMBA PERFORMANCE BY ECTOPIC BURDEN =====
        Ectopy group  AF N  NSR N      AUROC (95% CI)      AUPRC (95% CI)
         Low (<0.1%)     4   1021 0.842 [0.757-0.944] 0.017 [0.013-0.041]
Moderate (0.1-<0.5%)     5    211 0.854 [0.667-0.973] 0.189 [0.062-0.590]
       High (>=0.5%)     3     94 0.933 [0.793-1.000] 0.603 [0.129-1.000]
             Overall    12   1326 0.887 [0.785-0.978] 0.207 [0.054-0.490]


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

CSV_PATH = Path("Results/gee_oof_mamba_hrv.csv")

df = pd.read_csv(CSV_PATH)
df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

if "mamba_01" not in df.columns:
    df["mamba_01"] = df["mamba"] * 10

FORMULAS = {
    "AF-Mamba only": "label ~ mamba_01 + C(fold)",
    "+ RMSSD": "label ~ mamba_01 + rmssd + C(fold)",
    "+ SD1/SD2": "label ~ mamba_01 + sd1sd2 + C(fold)",
    "+ SODP-Q1": "label ~ mamba_01 + q1 + C(fold)",
    "+ SODP-CTM100": "label ~ mamba_01 + ctm + C(fold)",
    "+ PAS": "label ~ mamba_01 + pas + C(fold)",
    "+ all five HRV measures":
        "label ~ mamba_01 + rmssd + sd1sd2 + q1 + ctm + pas + C(fold)"
}

results = []

for name, formula in FORMULAS.items():
    fit = smf.gee(
        formula=formula,
        groups="subject_id",
        data=df,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Independence()
    ).fit(maxiter=500)

    ci = fit.conf_int().loc["mamba_01"]

    results.append({
        "Model": name,
        "OR": np.exp(fit.params["mamba_01"]),
        "CI Low": np.exp(ci.iloc[0]),
        "CI High": np.exp(ci.iloc[1]),
        "P": fit.pvalues["mamba_01"]
    })

results = pd.DataFrame(results)

print("\n===== HRV-ADJUSTED GEE RESULTS =====")
for _, r in results.iterrows():
    p = "<0.001" if r["P"] < 0.001 else f'{r["P"]:.3f}'
    print(
        f'{r["Model"]}: '
        f'OR={r["OR"]:.2f} '
        f'({r["CI Low"]:.2f}-{r["CI High"]:.2f}), '
        f'p={p}'
    )


===== HRV-ADJUSTED GEE RESULTS =====
AF-Mamba only: OR=2.41 (2.15-2.71), p=<0.001
+ RMSSD: OR=2.49 (2.19-2.83), p=<0.001
+ SD1/SD2: OR=2.13 (1.89-2.39), p=<0.001
+ SODP-Q1: OR=2.41 (2.15-2.70), p=<0.001
+ SODP-CTM100: OR=2.43 (2.13-2.78), p=<0.001
+ PAS: OR=2.42 (2.16-2.71), p=<0.001
+ all five HRV measures: OR=2.20 (1.92-2.52), p=<0.001
